# 02 — Split design and leakage controls

This notebook documents the frozen MonReader evaluation populations. Split construction lives in `src/splits.py`; this notebook reads the committed registrations, validates the frozen manifests, and explains which generalization question each split answers.


In [1]:
from pathlib import Path
import csv
import json
import sys
import yaml

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Run from the repository root or notebooks/ directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.splits import validate_dataset_split_compatibility, validate_split

SPLIT_DIR = ROOT / "manifests" / "splits"


## Reviewed video → environment/session map

The conservative minimum grouping unit is the canonical video. A stricter environment/session map was reviewed from early/middle/late frames for every video. Videos were merged only where repeated book/document identity, layout, camera/table geometry, hands, lighting, and numbering context supported the relationship. Ambiguous relations remain separate by default.


In [2]:
environment_review = yaml.safe_load((ROOT / "manifests" / "environment_groups.yaml").read_text(encoding="utf-8"))
video_map = list(csv.DictReader((ROOT / "manifests" / "video_environment_map.csv").open(newline="", encoding="utf-8")))
print(f"reviewed videos: {len(video_map)}")
print(f"environment/session groups: {len(environment_review['groups'])}")
for group in environment_review["groups"]:
    print(f"{group['environment_id']}: {group['description']} — {group['members']}")


reviewed videos: 117
environment/session groups: 4
ENV-01: technical/workbook session A — flip/0001–0010; notflip/0001–0011
ENV-02: small prose paperback session — flip/0011–0033; notflip/0015–0026
ENV-03: illustrated Stephen Hawking paperback session — flip/0034–0048; notflip/0027–0040
ENV-04: technical/reference book session B — flip/0049–0065; notflip/0041–0058


## Frozen split registrations

The five public-facing regimes are **Original**, **Random Stratified**, **Source-Safe**, **Temporal-Ordered**, and **Source-Safe + Temporal**. A separate **Video-Disjoint** registration is retained so the effect of broader environment/session grouping can be measured rather than hidden.


In [3]:
split_summary = json.loads((SPLIT_DIR / "split_summary.json").read_text(encoding="utf-8"))
for split_id, registration in split_summary["splits"].items():
    print(split_id, registration["name"])
    print("  partitions:", registration["summary"]["partition_counts"])
    print("  labels:", registration["summary"]["partition_label_counts"])


SPLIT-001 Original
  partitions: {'test': 597, 'train': 2392}
  labels: {'test': {'flip': 290, 'notflip': 307}, 'train': {'flip': 1162, 'notflip': 1230}}
SPLIT-002 Random Stratified
  partitions: {'test': 301, 'train': 2390, 'validation': 298}
  labels: {'test': {'flip': 146, 'notflip': 155}, 'train': {'flip': 1161, 'notflip': 1229}, 'validation': {'flip': 145, 'notflip': 153}}
SPLIT-003 Video-Disjoint
  partitions: {'test': 298, 'train': 2396, 'validation': 295}
  labels: {'test': {'flip': 145, 'notflip': 153}, 'train': {'flip': 1162, 'notflip': 1234}, 'validation': {'flip': 145, 'notflip': 150}}
SPLIT-004 Source-Safe
  partitions: {'test': 770, 'train': 1635, 'validation': 584}
  labels: {'test': {'flip': 348, 'notflip': 422}, 'train': {'flip': 855, 'notflip': 780}, 'validation': {'flip': 249, 'notflip': 335}}
SPLIT-005 Temporal-Ordered
  partitions: {'excluded_short_video': 332, 'test': 283, 'train': 2109, 'validation': 265}
  labels: {'excluded_short_video': {'flip': 324, 'notflip'

## Leakage and chronology checks

Each frozen CSV is revalidated below. Video-disjoint regimes may not place one canonical video in multiple partitions. Source-Safe also requires environment/session disjointness. Temporal-Ordered requires ascending `FrameNumber` with earliest 80% / middle 10% / latest 10% inside each video with at least 20 frames. Shorter videos are excluded from the primary temporal population. Source-Safe + Temporal uses one complete environment/session as the outer test holdout and chronological 90/10 fitting/validation inside the remaining eligible videos.


In [4]:
for split_id, registration in split_summary["splits"].items():
    path = ROOT / registration["manifest"]["path"]
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    validate_split(rows, split_name=registration["name"])
    print(f"PASS {registration['name']}: {len(rows):,} rows")


PASS Original: 2,989 rows
PASS Random Stratified: 2,989 rows
PASS Video-Disjoint: 2,989 rows
PASS Source-Safe: 2,989 rows
PASS Temporal-Ordered: 2,989 rows
PASS Source-Safe + Temporal: 2,989 rows


## Source-safe hierarchy

`Video-Disjoint` is the minimum source-safe construction. `Source-Safe` is stricter: the four reviewed environment/session groups are kept disjoint. The final test environment is selected deterministically for class balance, with validation selected the same way from the remaining groups. Because only four broader groups are defensible, their partition sizes are necessarily coarser than an 80/10/10 frame split.


In [5]:
source_safe = split_summary["splits"]["SPLIT-004"]
print(yaml.safe_dump(source_safe["rule"], sort_keys=False))


environment_partition:
  ENV-01: validation
  ENV-02: train
  ENV-03: test
  ENV-04: train
group: environment_id
selection: test environment closest to global class proportion; validation chosen
  likewise from remaining groups
type: environment_session_disjoint



## Temporal population

The primary temporal analysis excludes videos with fewer than 20 frames rather than pooling them into the main result. The exclusion remains explicit in the split manifest so the evaluation population is auditable.


In [6]:
temporal = split_summary["splits"]["SPLIT-005"]
combined = split_summary["splits"]["SPLIT-006"]
print("Temporal-Ordered:", temporal["summary"]["partition_counts"])
print("Source-Safe + Temporal:", combined["summary"]["partition_counts"])


Temporal-Ordered: {'excluded_short_video': 332, 'test': 283, 'train': 2109, 'validation': 265}
Source-Safe + Temporal: {'excluded_short_video': 293, 'test': 770, 'train': 1723, 'validation': 203}


## Dataset / split compatibility

The command-line entrypoint validates dataset/split compatibility before execution. This prevents a later experiment from silently pairing a registered split with the wrong dataset variant.


In [7]:
compatibility = ROOT / "manifests" / "dataset_split_compatibility.yaml"
for split_id in split_summary["splits"]:
    validate_dataset_split_compatibility(compatibility, dataset_id="DATA-001", split_id=split_id)
print("PASS: all frozen Phase-3 splits are compatible with the canonical original-RGB dataset")


PASS: all frozen Phase-3 splits are compatible with the canonical original-RGB dataset


## Phase-3 conclusion

The evaluation populations are frozen before model development expands. The supplied split remains a benchmark reference, frame-random performance is separated from source-safe performance, broader grouping is explicit and reviewable, chronology is enforced centrally, and exact decoded-pixel duplicate leakage is checked against the canonical dataset hashes.
